In [7]:
#@title setup
!pip install -q datasets trafilatura justext fasttext huggingface_hub transformers torch pandas

import html
from bs4 import BeautifulSoup
from IPython.display import display, HTML
from datasets import load_dataset
import pandas as pd
import os
import json
from datasets import load_dataset
import urllib.request
import textwrap
from torch.utils.data import IterableDataset, DataLoader



# 1) Common Crawl Samples

In [ ]:
common_crawl_dataset = load_dataset("agentlans/common-crawl-sample",
                                    split="train",
                                    streaming=True)

target_snippets = [
    "christianity.stackexchange.com/questions/1977",
    "slideshare.net",
    "englewoodwater.com"
]

matched_records = []
scanned = 0
max_search = 5000

print("Searching dataset for target URLs using substring matching...")
for record in common_crawl_dataset:
    scanned += 1
    if scanned % 1000 == 0:
        print(f"Scanned {scanned} records...")

    url = record.get("url", record.get("uri", ""))
    if any(snippet in url for snippet in target_snippets):
        record["url"] = url  # Ensure the 'url' key exists for the DataFrame
        matched_records.append(record)
        print(f"Found: {url}")

    if len(matched_records) >= len(target_snippets):
        print("All targets found!")
        break

    if scanned >= max_search:
        print(f"Reached maximum search limit ({max_search}) without finding all URLs.")
        break

if matched_records:
    cc_samples_df = pd.DataFrame(matched_records)
    display(cc_samples_df[["url", "text"]].style.set_properties(**{"white-space": "pre-wrap", "text-align": "left", "max-width": "600px"}))
else:
    print("No records matched the target URLs in the scanned sample.")

Searching dataset for target URLs using substring matching...
Found: http://englewoodwater.com/
Found: http://www.slideshare.net/errino/project-server-in-the-oil-and-gas-industry-enabling-technologies-best-practices
Found: http://christianity.stackexchange.com/questions/1977/is-it-hypocritical-to-be-a-christian-and-also-be-against-welfare/2055
All targets found!


,url,text
0,http://englewoodwater.com/,The District encompasses approximately 44.5 square miles in Southern Sarasota County and Western Charlotte County. There are four (4) fresh water and two (2) brackish water wellfields. This raw water is treated at either the lime softening treatment plant or the reverse osmosis treatment plant. The finished waters are mixed and sent into the distribution system. The District also operates a 3.0 MGD Water Reclamation Facility to take care of the sanitary sewer needs of the area. As of September 2014 84% of all water customers also receive sewer service. 1955
1,http://www.slideshare.net/errino/project-server-in-the-oil-and-gas-industry-enabling-technologies-best-practices,"support management labor costs in the energy of project portfolios, such systems were often difficult to use andlacked critical business (up- and downstream) functionality. A significant disconnect remains in the availability andexchange of management information between the portfolio and project management functions.The effect of this disconnect is an inability of the portfolio managers to track the actual status of the portfolio andto support investment decision making. It also leads to a gap in the availability of aggregated managementinformation at the central “projects office” level, making the review of projects underway a more time-consumingfunction and preventing management from intervening quickly when projects require additional support or start tofail.A company that does not successfully link project management with program and ultimately portfoliomanagement may notice that it is unable to:  Identify and prioritize successful and profitable project opportunities at regional, business unit, and corporate level;  Monitor financial thresholds of portfolio as well as earned value;  Maintain a balanced portfolio as circumstances and realities change;  Develop appropriate contracting strategies to reflect the technical, geographical, and commercial profile of the project pipeline;  Monitor and report the delivery performance of the existing portfolio;  Identify limited or critical project-delivery resources at the portfolio level and optimize these across a number of projects.A common result of organizations that fail to link project and portfolio data is the regular emergence of “surprises”on project overruns at the portfolio level – often too late for any meaningful management intervention and aftersuccessful time-scope-quality project delivery has been communicated to executive management.However, software is already available that provides visibility of all critical project and portfolio data and keyperformance indicators (KPIs) while enabling organization to make best practices accessible to all business units inthe Oil & Gas sector. By supporting full collaboration between all contributors to a venture, these systems fromestablished vendors allow companies to integrate information from project to portfolio level, and replicateprocesses that have been proven to work within different lines of business. Moreover, the data standardizationthat is the natural result of implementing integrated systems means that PPM decision makers can make ""apples-to-apples"" comparisons that facilitate prioritization of projects in a way that maximizes profitability and minimizesrisk.Here is a sampling of how Microsoft Project Server 2010 can address some of the more common PPM challengeswithin the context of a best-practices framework.  Balance resource demands with availability. Resource availability – or having the right number of people with the right skills in the right place at the right time – is a major issue in the Oil & Gas sector. The stiff competition for resources -- which happens even within the same company -- inevitably drives prices up. Companies implementing systems that give them the ability to forecast resource demands are able to estimate any shortfalls and develop strategies for bringing in additional r

In [ ]:
from IPython.display import display, HTML

iframe_urls = [
    "https://englewoodwater.com/",
]

for url in iframe_urls:
    display(HTML(f'<h4>{url}</h4><iframe src="{url}" width="100%" height="400" style="border:1px solid #ccc; border-radius:4px;"></iframe>'))

# 2) HTML Is Messy

In [ ]:
raw_html_response = urllib.request.urlopen("https://englewoodwater.com/")
raw_html_bytes = raw_html_response.read()
raw_html_text = raw_html_bytes.decode("utf-8", errors="replace")

# Use BeautifulSoup to find the specific divs
soup = BeautifulSoup(raw_html_text, "html.parser")
target_divs = soup.select('div#Content[role="main"]')

if target_divs:
    extracted_html = "\n\n".join(str(div) for div in target_divs)
else:
    extracted_html = "No matching divs found."

# Save to a local text file
with open("target_divs.html", "w", encoding="utf-8") as f:
    f.write(extracted_html)

# Escape the HTML so it displays as raw code instead of rendering
escaped_html = html.escape(extracted_html)
display(HTML(f'<div style="max-height: 500px; overflow-y: auto; background: #f5f5f5; padding: 10px; border: 1px solid #ccc;"><pre style="white-space: pre-wrap; word-wrap: break-word;">{escaped_html}</pre></div>'))
print(f"\n... ({len(extracted_html):,} total characters in extracted divs saved to 'target_divs.html')")


... (17,135 total characters in extracted divs saved to 'target_divs.html')


# 3) Clean Up the HTML

### 3.1) Trafilatura Extraction

In [ ]:
import trafilatura
import textwrap

trafilatura_extracted = trafilatura.extract(raw_html_text)
print("=" * 80)
print("TRAFILATURA OUTPUT")
print("=" * 80)
if trafilatura_extracted:
    print(textwrap.fill(trafilatura_extracted[:2000], width=80))
else:
    print("No content extracted")
print()

TRAFILATURA OUTPUT
Welcome to Englewood Water District Delivering High Quality, Clean Water While
Protecting Our Environment Englewood Water District (Englewood, FL) The District
encompasses approximately 44.5 square miles in Southern Sarasota County and
Western Charlotte County. There are four (4) fresh water and two (2) brackish
water wellfields. This raw water is treated at either the lime softening
treatment plant or the reverse osmosis treatment plant. The finished waters are
mixed and sent into the distribution system. The District also operates a 3.0
MGD Water Reclamation Facility to take care of the sanitary sewer needs of the
area. As of January 2019 87% of all water customers also receive sewer service.
In 2023 customers within the Englewood Water District service area used an
estimated 75 gallons per capita per day (gpcd). To see how your water usage
stacks up please visit SWFWMD Water Use Calculator.



### 3.2) Justext Extraction

In [ ]:
import justext
import textwrap

justext_paragraphs = justext.justext(raw_html_bytes, justext.get_stoplist("English"))
justext_extracted = "\n".join([p.text for p in justext_paragraphs if not p.is_boilerplate])
print("=" * 80)
print("JUSTEXT OUTPUT")
print("=" * 80)
if justext_extracted:
    print(textwrap.fill(justext_extracted[:2000], width=80))
else:
    print("No content extracted")
print()

JUSTEXT OUTPUT
Assistance paying utility bills may be available. Sarasota County residents can
view information on the Emergency Rental Assistance Program here. Charlotte
County residents can view information on the Fastrack Program here. Reminder —
Please do not flush items down toilets that do not belong there. Items such as
Paper Towels, Wipes (such as Baby Wipes, Disinfecting Wipes, etc..) and Cloth
Rags can cause problems for sewer collection systems. These items can also cause
blockages in a households own sewer pipes, which could cause sewer to back up
into peoples homes. Items such as these should be discarded in the trash.
Wastewater that comes from flushing should only contain water, human waste and
toilet paper. Englewood Water District (Englewood, FL) The District encompasses
approximately 44.5 square miles in Southern Sarasota County and Western
Charlotte County. There are four (4) fresh water and two (2) brackish water
wellfields. This raw water is treated at either the l

### 3.3) Naive HTML Stripper

In [ ]:
from html.parser import HTMLParser
import textwrap

class NaiveHTMLStripper(HTMLParser):
    def __init__(self):
        super().__init__()
        self.fed = []
        self.skip_tags = {"script", "style", "noscript"}
        self.current_skip = False
    def handle_starttag(self, tag, attrs):
        if tag in self.skip_tags:
            self.current_skip = True
    def handle_endtag(self, tag):
        if tag in self.skip_tags:
            self.current_skip = False
    def handle_data(self, data):
        if not self.current_skip:
            self.fed.append(data)
    def get_text(self):
        return " ".join("".join(self.fed).split())

naive_stripper = NaiveHTMLStripper()
naive_stripper.feed(raw_html_text)
naive_stripped_text = naive_stripper.get_text()

print("=" * 80)
print("NAIVE STRIP-TAGS OUTPUT (simulates raw WET file)")
print("=" * 80)
print(textwrap.fill(naive_stripped_text[:2000], width=80))
print()

NAIVE STRIP-TAGS OUTPUT (simulates raw WET file)
Englewood Water District - Englewood FL 941-474-3217866-460-1080 Home CodeRED
About Us Administration Board of Supervisors Board of Supervisors Meetings
Employee Benefits Committee Meetings Biography Supervisor Election District Maps
Important Documents Customer Service Hurricane Frequently Asked Questions How to
check for a leak Reading Your Bill Reading the meter Disconnect Policy Water
Restrictions Rates Finance Financial Information Defined Benefit Pension Report
EWD Actuarial Fact Sheet MyFlorida Actuarial Fact Sheet Operations Laboratory
Drinking Water Standards Annual Water Report Water Production Well Fields
Reverse Osmosis Plant Lime Softening Plant Distribution Chromium: Frequently
Asked Questions Wastewater Treatment Collections System Reclamation Plant
Reclaimed Water System Reclaimed Water Frequently Asked questions (FAQ)
Engineering Developer’s Info Project Preliminary Submittal List Project
Certification Submittal List Pro

### 3.4) Side-by-Side Comparison

In [ ]:
from IPython.display import display, HTML
import html

# Safely extract a portion of text for comparison
t_text = html.escape((trafilatura_extracted or "")[:1500]) + ("..." if trafilatura_extracted and len(trafilatura_extracted) > 1500 else "")
j_text = html.escape((justext_extracted or "")[:1500]) + ("..." if justext_extracted and len(justext_extracted) > 1500 else "")
n_text = html.escape((naive_stripped_text or "")[:1500]) + ("..." if naive_stripped_text and len(naive_stripped_text) > 1500 else "")

table_html = f"""
<table style="width:100%; text-align:left; border-collapse: collapse; border: 1px solid #ccc;">
    <tr style="background-color: #f2f2f2;">
        <th style="padding: 10px; border: 1px solid #ccc; width: 33%;">Trafilatura</th>
        <th style="padding: 10px; border: 1px solid #ccc; width: 33%;">Justext</th>
        <th style="padding: 10px; border: 1px solid #ccc; width: 33%;">Naive HTML Stripper</th>
    </tr>
    <tr>
        <td style="padding: 10px; border: 1px solid #ccc; vertical-align: top;"><pre style="white-space: pre-wrap; word-wrap: break-word; font-size: 12px; font-family: monospace;">{t_text}</pre></td>
        <td style="padding: 10px; border: 1px solid #ccc; vertical-align: top;"><pre style="white-space: pre-wrap; word-wrap: break-word; font-size: 12px; font-family: monospace;">{j_text}</pre></td>
        <td style="padding: 10px; border: 1px solid #ccc; vertical-align: top;"><pre style="white-space: pre-wrap; word-wrap: break-word; font-size: 12px; font-family: monospace;">{n_text}</pre></td>
    </tr>
</table>
"""

display(HTML(table_html))

Trafilatura,Justext,Naive HTML Stripper
"Welcome to Englewood Water District Delivering High Quality, Clean Water While Protecting Our Environment Englewood Water District (Englewood, FL) The District encompasses approximately 44.5 square miles in Southern Sarasota County and Western Charlotte County. There are four (4) fresh water and two (2) brackish water wellfields. This raw water is treated at either the lime softening treatment plant or the reverse osmosis treatment plant. The finished waters are mixed and sent into the distribution system. The District also operates a 3.0 MGD Water Reclamation Facility to take care of the sanitary sewer needs of the area. As of January 2019 87% of all water customers also receive sewer service. In 2023 customers within the Englewood Water District service area used an estimated 75 gallons per capita per day (gpcd). To see how your water usage stacks up please visit SWFWMD Water Use Calculator.","Assistance paying utility bills may be available. Sarasota County residents can view information on the Emergency Rental Assistance Program here. Charlotte County residents can view information on the Fastrack Program here. Reminder — Please do not flush items down toilets that do not belong there. Items such as Paper Towels, Wipes (such as Baby Wipes, Disinfecting Wipes, etc..) and Cloth Rags can cause problems for sewer collection systems. These items can also cause blockages in a households own sewer pipes, which could cause sewer to back up into peoples homes. Items such as these should be discarded in the trash. Wastewater that comes from flushing should only contain water, human waste and toilet paper. Englewood Water District (Englewood, FL) The District encompasses approximately 44.5 square miles in Southern Sarasota County and Western Charlotte County. There are four (4) fresh water and two (2) brackish water wellfields. This raw water is treated at either the lime softening treatment plant or the reverse osmosis treatment plant. The finished waters are mixed and sent into the distribution system. The District also operates a 3.0 MGD Water Reclamation Facility to take care of the sanitary sewer needs of the area. As of January 2019 87% of all water customers also receive sewer service. In 2023 customers within the Englewood Water District service area used an estimated 75 gallons per capita per day (gpcd). To see how your water usage stacks up please visit SWFWMD Wat...",Englewood Water District - Englewood FL 941-474-3217866-460-1080 Home CodeRED About Us Administration Board of Supervisors Board of Supervisors Meetings Employee Benefits Committee Meetings Biography Supervisor Election District Maps Important Documents Customer Service Hurricane Frequently Asked Questions How to check for a leak Reading Your Bill Reading the meter Disconnect Policy Water Restrictions Rates Finance Financial Information Defined Benefit Pension Report EWD Actuarial Fact Sheet MyFlorida Actuarial Fact Sheet Operations Laboratory Drinking Water Standards Annual Water Report Water Production Well Fields Reverse Osmosis Plant Lime Softening Plant Distribution Chromium: Frequently Asked Questions Wastewater Treatment Collections System Reclamation Plant Reclaimed Water System Reclaimed Water Frequently Asked questions (FAQ) Engineering Developer’s Info Project Preliminary Submittal List Project Certification Submittal List Project Submittal Form Details Construction Standards EWD Public Web Map EWD Facilities Plan Master Plans 2024 Potable Water Master Plan Update Potable Water Master Plan Update Appendix A Appendix B Appendix C Appendix D 2022 Reuse Master Plan Update Reuse Master Plan Update 2021 Sewer Master Plan Update Sewer Master Plan Update Executive Summary Appendix A Appendix B Appendix C Appendix D 2017 Utility Master Plan Utility Master Plan Final Appendix A Appendix B Appendix C Appendix D Appendix E Purchasing Forms Assessment Pay Off Autopay Applicati...


# 4) FineWeb-Edu Samples (Streaming)

In [ ]:
from datasets import load_dataset
import pandas as pd

fineweb_edu_stream = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-100BT",
    split="train",
    streaming=True,
)

fineweb_edu_samples = []
for record in fineweb_edu_stream:
    fineweb_edu_samples.append(record)
    if len(fineweb_edu_samples) == 3:
        break

fineweb_edu_df = pd.DataFrame(fineweb_edu_samples)
display(
    fineweb_edu_df[["url", "text", "score"]]
    .style.set_properties(**{"white-space": "pre-wrap", "text-align": "left", "max-width": "600px"})
    .set_properties(subset=["url"], **{"width": "10%"})
)

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/140 [00:00<?, ?it/s]

# 5) Quality Filtering: fastText DCLM Classifier

In [ ]:
import fasttext
from huggingface_hub import hf_hub_download

dclm_model_path = hf_hub_download(
    repo_id="mlfoundations/fasttext-oh-eli5",
    filename="openhermes_reddit_eli5_vs_rw_v2_bigram_200k_train.bin",
)

dclm_classifier = fasttext.load_model(dclm_model_path)

def dclm_quality_score(text, classifier=dclm_classifier):
    cleaned = text.replace("\n", " ").strip()[:5000]
    # Bypass the buggy wrapper to avoid NumPy 2.0 error
    predictions = classifier.f.predict(cleaned, 1, 0.0, "")
    probability, label = predictions[0]
    if label == "__label__hq":
        return probability
    else:
        return 1.0 - probability

print("=" * 60)
print("DCLM SCORES — Common Crawl Samples")
print("=" * 60)
for _, row in cc_samples_df.iterrows():
    score = dclm_quality_score(row["text"])
    print(f"  {score:.4f}  {row['url'][:80]}")

print()
print("=" * 60)
print("DCLM SCORES — FineWeb-Edu Samples")
print("=" * 60)
for _, row in fineweb_edu_df.iterrows():
    score = dclm_quality_score(row["text"])
    print(f"  {score:.4f}  {row['url'][:80]}")

DCLM SCORES — Common Crawl Samples
  -0.0000  http://englewoodwater.com/
  0.0003  http://www.slideshare.net/errino/project-server-in-the-oil-and-gas-industry-enab
  0.5719  http://christianity.stackexchange.com/questions/1977/is-it-hypocritical-to-be-a-

DCLM SCORES — FineWeb-Edu Samples
  0.0278  http://austenauthors.net/the-independent-jane
  0.2146  http://cloudyandcool.com/2009/05/05/wind-shear-and-tornadoes/
  0.0083  http://cogweb.ucla.edu/ep/FluteDebate.html


# 6) Quality Filtering: FineWeb-Edu Classifier

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

fineweb_classifier_name = "HuggingFaceFW/fineweb-edu-classifier"
fineweb_tokenizer = AutoTokenizer.from_pretrained(fineweb_classifier_name)
fineweb_quality_model = AutoModelForSequenceClassification.from_pretrained(fineweb_classifier_name)
fineweb_quality_model.eval()

def fineweb_edu_quality_score(text, tokenizer=fineweb_tokenizer, model=fineweb_quality_model):
    truncated = text[:2000]
    tokens = tokenizer(truncated, return_tensors="pt", padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        score = model(**tokens).logits.squeeze(-1).item()
    return round(score, 4)

print("=" * 60)
print("FINEWEB-EDU SCORES — Common Crawl Samples")
print("=" * 60)
for _, row in cc_samples_df.iterrows():
    score = fineweb_edu_quality_score(row["text"])
    print(f"  {score:.4f}  {row['url'][:80]}")

print()
print("=" * 60)
print("FINEWEB-EDU SCORES — FineWeb-Edu Samples")
print("=" * 60)
for _, row in fineweb_edu_df.iterrows():
    score = fineweb_edu_quality_score(row["text"])
    print(f"  {score:.4f}  {row['url'][:80]}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

FINEWEB-EDU SCORES — Common Crawl Samples
  1.6884  http://englewoodwater.com/
  1.1438  http://www.slideshare.net/errino/project-server-in-the-oil-and-gas-industry-enab
  1.1081  http://christianity.stackexchange.com/questions/1977/is-it-hypocritical-to-be-a-

FINEWEB-EDU SCORES — FineWeb-Edu Samples
  2.8415  http://austenauthors.net/the-independent-jane
  4.0641  http://cloudyandcool.com/2009/05/05/wind-shear-and-tornadoes/
  3.5170  http://cogweb.ucla.edu/ep/FluteDebate.html


# 7) Preparing N+1 pairs for training

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokens = tokenizer(justext_extracted)["input_ids"]

inputs = []
targets = []

# Group into small sequences (e.g., 7 tokens) for demonstration
seq_length = 7
for i in range(0, len(tokens) - seq_length, seq_length):
    # Grab a chunk of size seq_length + 1
    chunk = tokens[i : i + seq_length + 1]

    if len(chunk) == seq_length + 1:
        # Input is everything except the last token
        inputs.append(chunk[:-1])
        # Target is everything except the first token (shifted by 1)
        targets.append(chunk[1:])


print("--- First Pair ---")
print(f"Input IDs:  {inputs[0]}")
print(f"Target IDs: {targets[0]}\n")

print(f"Decoded Input:  {repr(tokenizer.decode(inputs[0]))}")
print(f"Decoded Target: {repr(tokenizer.decode(targets[0]))}")


--- First Pair ---
Input IDs:  [8021, 9311, 5989, 10361, 9024, 743, 307]
Target IDs: [9311, 5989, 10361, 9024, 743, 307, 1695]

Decoded Input:  'Assistance paying utility bills may be'
Decoded Target: 'istance paying utility bills may be available'


# 7) Exercise: fetch a website's HTML, try all three text extractors, and compare quality

In [ ]:
#@title Answer
# 1. Define the target URL and fetch the HTML
target_url = "https://en.wikipedia.org/wiki/Web_scraping"
print(f"Fetching HTML from: {target_url}")

req = urllib.request.Request(target_url, headers={'User-Agent': 'Mozilla/5.0'})
with urllib.request.urlopen(req) as response:
    raw_bytes = response.read()
    raw_text = raw_bytes.decode('utf-8', errors='replace')

# 2. Extract text using Trafilatura
print("Extracting using Trafilatura...")
ext_traf = trafilatura.extract(raw_text) or "No content extracted"

# 3. Extract text using Justext
print("Extracting using Justext...")
paragraphs = justext.justext(raw_bytes, justext.get_stoplist("English"))
ext_just = "\n".join([p.text for p in paragraphs if not p.is_boilerplate]) or "No content extracted"

# 4. Extract text using the Naive HTML Stripper defined earlier in the notebook
print("Extracting using Naive HTML Stripper...")
my_stripper = NaiveHTMLStripper()
my_stripper.feed(raw_text)
ext_naive = my_stripper.get_text()

# 5. Quality Filtering
print("Calculating quality scores...")
traf_dclm = dclm_quality_score(ext_traf)
just_dclm = dclm_quality_score(ext_just)
naive_dclm = dclm_quality_score(ext_naive)

traf_fw = fineweb_edu_quality_score(ext_traf)
just_fw = fineweb_edu_quality_score(ext_just)
naive_fw = fineweb_edu_quality_score(ext_naive)

# 6. Display a side-by-side comparison table
print("Generating comparison table...\n")
def truncate_and_escape(text, length=200):
    safe_text = html.escape((text or "")[:length])
    return safe_text + ("..." if len(text or "") > length else "")

table_html = f"""
<table style="width:100%; text-align:left; border-collapse: collapse; border: 1px solid #ccc; table-layout: fixed;">
    <tr style="background-color: #f2f2f2;">
        <th style="padding: 10px; border: 1px solid #ccc; width: 33.33%;">Trafilatura</th>
        <th style="padding: 10px; border: 1px solid #ccc; width: 33.33%;">Justext</th>
        <th style="padding: 10px; border: 1px solid #ccc; width: 33.33%;">Naive HTML Stripper</th>
    </tr>
    <tr>
        <td style="padding: 10px; border: 1px solid #ccc; vertical-align: top;"><pre style="white-space: pre-wrap; word-wrap: break-word; font-size: 12px; font-family: monospace;">{truncate_and_escape(ext_traf)}</pre></td>
        <td style="padding: 10px; border: 1px solid #ccc; vertical-align: top;"><pre style="white-space: pre-wrap; word-wrap: break-word; font-size: 12px; font-family: monospace;">{truncate_and_escape(ext_just)}</pre></td>
        <td style="padding: 10px; border: 1px solid #ccc; vertical-align: top;"><pre style="white-space: pre-wrap; word-wrap: break-word; font-size: 12px; font-family: monospace;">{truncate_and_escape(ext_naive)}</pre></td>
    </tr>
    <tr>
        <td style="padding: 10px; border: 1px solid #ccc;"><b>Characters:</b> {len(ext_traf):,}</td>
        <td style="padding: 10px; border: 1px solid #ccc;"><b>Characters:</b> {len(ext_just):,}</td>
        <td style="padding: 10px; border: 1px solid #ccc;"><b>Characters:</b> {len(ext_naive):,}</td>
    </tr>
    <tr>
        <td style="padding: 10px; border: 1px solid #ccc;"><b>DCLM Score:</b> {traf_dclm:.4f}</td>
        <td style="padding: 10px; border: 1px solid #ccc;"><b>DCLM Score:</b> {just_dclm:.4f}</td>
        <td style="padding: 10px; border: 1px solid #ccc;"><b>DCLM Score:</b> {naive_dclm:.4f}</td>
    </tr>
    <tr>
        <td style="padding: 10px; border: 1px solid #ccc;"><b>FineWeb-Edu Score:</b> {traf_fw:.4f}</td>
        <td style="padding: 10px; border: 1px solid #ccc;"><b>FineWeb-Edu Score:</b> {just_fw:.4f}</td>
        <td style="padding: 10px; border: 1px solid #ccc;"><b>FineWeb-Edu Score:</b> {naive_fw:.4f}</td>
    </tr>
</table>
"""
display(HTML(table_html))


Fetching HTML from: https://en.wikipedia.org/wiki/Web_scraping
Extracting using Trafilatura...
Extracting using Justext...
Extracting using Naive HTML Stripper...
Calculating quality scores...
Generating comparison table...



Trafilatura,Justext,Naive HTML Stripper
"Web scraping This article needs additional citations for verification. (April 2023) | Web scraping, web harvesting, or web data extraction is data scraping used for extracting data from websites.[1] W...","Scraping a web page involves fetching it and then extracting data from it. Fetching is the downloading of a page (which a browser does when a user views a page). Therefore, web crawling is a main comp...",Web scraping - Wikipedia Jump to content Main menu Main menu move to sidebar hide Navigation Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us Contribute HelpLearn to editCommunit...
"Characters: 26,335","Characters: 15,787","Characters: 29,634"
DCLM Score: 0.2281,DCLM Score: 0.7749,DCLM Score: 0.0177
FineWeb-Edu Score: 3.4122,FineWeb-Edu Score: 3.8386,FineWeb-Edu Score: 2.0813


# 8) Sharding FineWeb-Edu for Local Storage (Faster Training)

In [4]:
shards_output_dir = "/mnt/user-data/outputs/shards"
os.makedirs(shards_output_dir, exist_ok=True)

# Configuration for our local shards
records_per_shard = 1000
max_shards = 5

fineweb_stream = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-100BT",
    split="train",
    streaming=True,
)

shard_index = 0
records_in_current_shard = 0
current_file = open(os.path.join(shards_output_dir, f"shard_{shard_index:03d}.jsonl"), "w", encoding="utf-8")

print(f"Writing to shard {shard_index}...")
for record in fineweb_stream:
    # Storing just the text and url for demonstration to keep sizes manageable
    out_record = {"url": record["url"], "text": record["text"]}
    current_file.write(json.dumps(out_record) + "\n")
    records_in_current_shard += 1

    # If the shard is full, close it and open the next one
    if records_in_current_shard >= records_per_shard:
        current_file.close()
        shard_index += 1

        if shard_index >= max_shards:
            break

        records_in_current_shard = 0
        print(f"Writing to shard {shard_index}...")
        current_file = open(os.path.join(shards_output_dir, f"shard_{shard_index:03d}.jsonl"), "w", encoding="utf-8")

if not current_file.closed:
    current_file.close()

print(f"\nSuccessfully saved {max_shards} shards with {records_per_shard} records each.")
print(f"Files saved to {shards_output_dir}/")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/140 [00:00<?, ?it/s]

Writing to shard 0...
Writing to shard 1...
Writing to shard 2...
Writing to shard 3...
Writing to shard 4...

Successfully saved 5 shards with 1000 records each.
Files saved to /mnt/user-data/outputs/shards/


# 9) Resuming 4,000 documents in, with binning

In [5]:
import os
import json

shard_index_to_read = 4
shard_file = os.path.join(shards_output_dir, f"shard_{shard_index_to_read:03d}.jsonl")

global_start_index = shard_index_to_read * records_per_shard

print(f"{shard_file} shard starts at global record {global_start_index + 1}\n")

with open(shard_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        record = json.loads(line)
        global_record_num = global_start_index + i + 1

        print(f"Global Record {global_record_num}:")
        print(f"URL: {record['url']}")
        print(f"Text Snippet: {record['text'][:300]}...\n")
        print("-" * 80 + "\n")

/mnt/user-data/outputs/shards/shard_004.jsonl shard starts at global record 4001

Global Record 4001:
URL: http://www.usaid.gov/results-data/success-stories/saving-newborns-rural-afghanistan
Text Snippet: Miriam has four children, although she has given birth six times. Two infants unfortunately died within days of birth. However, her most recent birth was different in that Miriam and her female relatives had learned from the local female community health worker (CHW) about the importance of keeping ...

--------------------------------------------------------------------------------

Global Record 4002:
URL: http://apnews.excite.com/article/20110219/D9LG45NO0.html
Text Snippet: Cosmic census finds crowd of planets in our galaxy
Feb 19, 5:23 PM (ET)
By SETH BORENSTEIN
WASHINGTON (AP) - Scientists have estimated the first cosmic census of planets in our galaxy and the numbers are astronomical: at least 50 billion planets in the Milky Way.
At least 500 million of those planet...

-------

# 10) Resuming 4000 documents in, with a custom DataLoader binning

In [10]:
class ShardDataset(Dataset):
    def __init__(self, shards_dir):
        self.shards_dir = shards_dir

    def __len__(self):
        return max_shards

    def __getitem__(self, idx):
        shard_file = os.path.join(self.shards_dir,
                                  f"shard_{idx:03d}.jsonl")
        records = []
        with open(shard_file, "r", encoding="utf-8") as f:
            for line in f:
                records.append(json.loads(line))
        return records

dataset = ShardDataset(shards_output_dir)

shard_index_to_read = 4
global_start_index = shard_index_to_read * records_per_shard

print(f"Reading shard {shard_index_to_read} via dataset[idx]... starts at global record {global_start_index + 1}\n")

# Retrieve all records for the specific shard file
shard_records = dataset[shard_index_to_read]

for i, record in enumerate(shard_records):
    global_record_num = global_start_index + i + 1

    print(f"Global Record {global_record_num}:")
    print(f"URL: {record['url']}")
    print(f"Text Snippet: {record['text'][:300]}...\n")
    print("-" * 80 + "\n")

Reading shard 4 via dataset[idx]... starts at global record 4001

Global Record 4001:
URL: http://www.usaid.gov/results-data/success-stories/saving-newborns-rural-afghanistan
Text Snippet: Miriam has four children, although she has given birth six times. Two infants unfortunately died within days of birth. However, her most recent birth was different in that Miriam and her female relatives had learned from the local female community health worker (CHW) about the importance of keeping ...

--------------------------------------------------------------------------------

Global Record 4002:
URL: http://apnews.excite.com/article/20110219/D9LG45NO0.html
Text Snippet: Cosmic census finds crowd of planets in our galaxy
Feb 19, 5:23 PM (ET)
By SETH BORENSTEIN
WASHINGTON (AP) - Scientists have estimated the first cosmic census of planets in our galaxy and the numbers are astronomical: at least 50 billion planets in the Milky Way.
At least 500 million of those planet...

-----------------------

# 11) Extract and quality score a web page

In [ ]:
assert False, "did you fetch your website, extracted HTML and compared quality?"

AssertionError: did you fetch your website, extracted HTML and compared quality?

In [14]:
#@title Answer (setup if you didn't want to run the whole notebook)

#@title Answer

import urllib.request
import html
from html.parser import HTMLParser
import trafilatura
import justext
from IPython.display import display, HTML
import fasttext
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

class NaiveHTMLStripper(HTMLParser):
    def __init__(self):
        super().__init__()
        self.fed = []
        self.skip_tags = {"script", "style", "noscript"}
        self.current_skip = False
    def handle_starttag(self, tag, attrs):
        if tag in self.skip_tags:
            self.current_skip = True
    def handle_endtag(self, tag):
        if tag in self.skip_tags:
            self.current_skip = False
    def handle_data(self, data):
        if not self.current_skip:
            self.fed.append(data)
    def get_text(self):
        return " ".join("".join(self.fed).split())

print("Loading quality classifiers...")
# Setup DCLM Classifier
dclm_model_path = hf_hub_download(
    repo_id="mlfoundations/fasttext-oh-eli5",
    filename="openhermes_reddit_eli5_vs_rw_v2_bigram_200k_train.bin",
)
dclm_classifier = fasttext.load_model(dclm_model_path)

def dclm_quality_score(text, classifier=dclm_classifier):
    cleaned = text.replace("\n", " ").strip()[:5000]
    predictions = classifier.f.predict(cleaned, 1, 0.0, "")
    probability, label = predictions[0]
    if label == "__label__hq":
        return probability
    else:
        return 1.0 - probability

# Setup FineWeb-Edu Classifier
fineweb_classifier_name = "HuggingFaceFW/fineweb-edu-classifier"
fineweb_tokenizer = AutoTokenizer.from_pretrained(fineweb_classifier_name)
fineweb_quality_model = AutoModelForSequenceClassification.from_pretrained(fineweb_classifier_name)
fineweb_quality_model.eval()

def fineweb_edu_quality_score(text, tokenizer=fineweb_tokenizer, model=fineweb_quality_model):
    truncated = text[:2000]
    tokens = tokenizer(truncated, return_tensors="pt", padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        score = model(**tokens).logits.squeeze(-1).item()
    return round(score, 4)

Loading quality classifiers...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [15]:
#@title Answer
# 1. Define the target URL and fetch the HTML
target_url = "https://en.wikipedia.org/wiki/Deep_learning"
print(f"\nFetching HTML from: {target_url}")

req = urllib.request.Request(target_url, headers={'User-Agent': 'Mozilla/5.0'})
with urllib.request.urlopen(req) as response:
    raw_bytes = response.read()
    raw_text = raw_bytes.decode('utf-8', errors='replace')

# 2. Extract text using Trafilatura
print("Extracting using Trafilatura...")
ext_traf = trafilatura.extract(raw_text) or "No content extracted"

# 3. Extract text using Justext
print("Extracting using Justext...")
paragraphs = justext.justext(raw_bytes, justext.get_stoplist("English"))
ext_just = "\n".join([p.text for p in paragraphs if not p.is_boilerplate]) or "No content extracted"

# 4. Extract text using the Naive HTML Stripper defined earlier
print("Extracting using Naive HTML Stripper...")
my_stripper = NaiveHTMLStripper()
my_stripper.feed(raw_text)
ext_naive = my_stripper.get_text()

# 5. Quality Filtering
print("Calculating quality scores...")
traf_dclm = dclm_quality_score(ext_traf)
just_dclm = dclm_quality_score(ext_just)
naive_dclm = dclm_quality_score(ext_naive)

traf_fw = fineweb_edu_quality_score(ext_traf)
just_fw = fineweb_edu_quality_score(ext_just)
naive_fw = fineweb_edu_quality_score(ext_naive)

# 6. Display a side-by-side comparison table
print("Generating comparison table...\n")
def truncate_and_escape(text, length=300):
    safe_text = html.escape((text or "")[:length])
    return safe_text + ("..." if len(text or "") > length else "")

table_html = f"""
<table style="width:100%; text-align:left; border-collapse: collapse; border: 1px solid #ccc; table-layout: fixed;">
    <tr style="background-color: #f2f2f2;">
        <th style="padding: 10px; border: 1px solid #ccc; width: 33.33%;">Trafilatura</th>
        <th style="padding: 10px; border: 1px solid #ccc; width: 33.33%;">Justext</th>
        <th style="padding: 10px; border: 1px solid #ccc; width: 33.33%;">Naive HTML Stripper</th>
    </tr>
    <tr>
        <td style="padding: 10px; border: 1px solid #ccc; vertical-align: top;"><pre style="white-space: pre-wrap; word-wrap: break-word; font-size: 12px; font-family: monospace;">{truncate_and_escape(ext_traf)}</pre></td>
        <td style="padding: 10px; border: 1px solid #ccc; vertical-align: top;"><pre style="white-space: pre-wrap; word-wrap: break-word; font-size: 12px; font-family: monospace;">{truncate_and_escape(ext_just)}</pre></td>
        <td style="padding: 10px; border: 1px solid #ccc; vertical-align: top;"><pre style="white-space: pre-wrap; word-wrap: break-word; font-size: 12px; font-family: monospace;">{truncate_and_escape(ext_naive)}</pre></td>
    </tr>
    <tr>
        <td style="padding: 10px; border: 1px solid #ccc;"><b>Characters:</b> {len(ext_traf):,}</td>
        <td style="padding: 10px; border: 1px solid #ccc;"><b>Characters:</b> {len(ext_just):,}</td>
        <td style="padding: 10px; border: 1px solid #ccc;"><b>Characters:</b> {len(ext_naive):,}</td>
    </tr>
    <tr>
        <td style="padding: 10px; border: 1px solid #ccc;"><b>DCLM Score:</b> {traf_dclm:.4f}</td>
        <td style="padding: 10px; border: 1px solid #ccc;"><b>DCLM Score:</b> {just_dclm:.4f}</td>
        <td style="padding: 10px; border: 1px solid #ccc;"><b>DCLM Score:</b> {naive_dclm:.4f}</td>
    </tr>
    <tr>
        <td style="padding: 10px; border: 1px solid #ccc;"><b>FineWeb-Edu Score:</b> {traf_fw:.4f}</td>
        <td style="padding: 10px; border: 1px solid #ccc;"><b>FineWeb-Edu Score:</b> {just_fw:.4f}</td>
        <td style="padding: 10px; border: 1px solid #ccc;"><b>FineWeb-Edu Score:</b> {naive_fw:.4f}</td>
    </tr>
</table>
"""
display(HTML(table_html))



Fetching HTML from: https://en.wikipedia.org/wiki/Deep_learning
Extracting using Trafilatura...
Extracting using Justext...
Extracting using Naive HTML Stripper...
Calculating quality scores...
Generating comparison table...



Trafilatura,Justext,Naive HTML Stripper
"Deep learning | Part of a series on | | Artificial intelligence (AI) | |---| In machine learning, deep learning (DL) focuses on utilizing multilayered neural networks to perform tasks such as classification, regression, and representation learning. The field takes inspiration from biological neurosc...","Early forms of neural networks were inspired by information processing and distributed communication nodes in biological systems, particularly the human brain. However, current neural networks do not intend to model the brain function of organisms, and are generally seen as low-quality models for th...",Deep learning - Wikipedia Jump to content Main menu Main menu move to sidebar hide Navigation Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us Contribute HelpLearn to editCommunity portalRecent changesUpload fileSpecial pages Search Search Appearance Donate Create account Log i...
"Characters: 122,136","Characters: 42,882","Characters: 133,899"
DCLM Score: 0.6824,DCLM Score: 0.4624,DCLM Score: 0.2187
FineWeb-Edu Score: 3.5533,FineWeb-Edu Score: 3.4824,FineWeb-Edu Score: 2.0283
